In [1]:
import os
import sys

In [3]:
os.getcwd()

'e:\\Udemy\\Project 4\\Insurance Fraud Detection\\notebooks'

In [9]:
try:
    10/0
except Exception as e:
    print(sys.exc_info())

(<class 'ZeroDivisionError'>, ZeroDivisionError('division by zero'), <traceback object at 0x000001BF76170E80>)


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option('display.max_columns', None)
from datetime import datetime

In [3]:
df = pd.read_excel(r"C:\Users\asus\Downloads\Worksheet in Case Study question 2.xlsx")

In [4]:
df1 = df.copy()

In [5]:
df1['collision_type'] = df1['collision_type'].replace('?','unknown') 
df1['property_damage'] = df1['property_damage'].replace('?','unknown') 
df1['police_report_available'] = df1['police_report_available'].replace('?', 'unknown') 
df1['authorities_contacted'] = df1['authorities_contacted'].fillna('unknown')

In [7]:
df1['policy_bind_date'] = pd.to_datetime(df1['policy_bind_date'])
df1['policy_bind_year'] = df1['policy_bind_date'].dt.year
df1['policy_bind_month'] = df1['policy_bind_date'].dt.month
df1['policy_bind_day'] = df['policy_bind_date'].dt.day
df1['incident_date'] = pd.to_datetime(df1['incident_date'])
df1['incident_month'] = df1['incident_date'].dt.month
df1['incident_day'] = df1['incident_date'].dt.day

In [8]:
df1.drop(['policy_bind_date','incident_date'], inplace=True, axis=1)

In [9]:
df1['fraud_reported'] = df1['fraud_reported'].map({
    "Y":1, "N":0
                                                   })

In [10]:
df1.drop("total_claim_amount", axis=1, inplace=True) # dropping as it being explained by other columns

In [11]:
df1.drop('policy_number', inplace=True, axis =1)
df1.drop('insured_zip', inplace=True,axis=1 )
df1.drop('incident_location',inplace=True, axis = 1 )

In [12]:
from sklearn.model_selection import train_test_split
X = df1.drop('fraud_reported', axis = 1)
y = df1['fraud_reported']

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [14]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (OneHotEncoder, StandardScaler, OrdinalEncoder)

In [15]:
one_hot_encoding_columns = ['policy_state','policy_csl', 'insured_occupation' , 'insured_hobbies', 
                            'insured_relationship','incident_type', 'collision_type', 'incident_severity','authorities_contacted', 
                             'incident_state', 'incident_city', 'auto_make', 'auto_model', ]
binary_encoding_columns = ['insured_sex', 'property_damage','police_report_available' ]
oridinal_encoding_columns = ['insured_education_level']
numrical_scaling_columns = ['months_as_customer', 'age', 'policy_deductable','policy_annual_premium', 'umbrella_limit', 'capital-gains',
                            'capital-loss', 'incident_hour_of_the_day', 'number_of_vehicles_involved','bodily_injuries','witnesses', 
                               'injury_claim', 'property_claim',   'vehicle_claim', 'auto_year', 'policy_bind_year', 'policy_bind_month',
                                 'policy_bind_day', 'incident_month','incident_day' ]

In [16]:
education_order = [['High School','Associate' ,'College','Masters', 'JD', 'MD','PhD']]

In [17]:
preprocessing =ColumnTransformer(transformers=[
    ('one-hot',OneHotEncoder(handle_unknown='ignore',drop='first',sparse_output=False),one_hot_encoding_columns),
    ('binary', OrdinalEncoder(),binary_encoding_columns),
    ('ordinal', OrdinalEncoder(categories=education_order),oridinal_encoding_columns),
    ('numerical', StandardScaler(),numrical_scaling_columns)
], remainder = 'drop')

In [18]:
categorical_columns = (
    one_hot_encoding_columns +
    binary_encoding_columns +
    oridinal_encoding_columns
)
X_train[categorical_columns] = X_train[categorical_columns].astype(str)
X_test[categorical_columns] = X_test[categorical_columns].astype(str)

In [37]:
for col in X_train.select_dtypes(include= 'object').columns.to_list():
    print (col,"-->" ,X_train[col].unique())

policy_state --> <StringArray>
['IN', 'OH', 'IL']
Length: 3, dtype: str
policy_csl --> <StringArray>
['250/500', '500/1000', '100/300']
Length: 3, dtype: str
insured_sex --> <StringArray>
['MALE', 'FEMALE']
Length: 2, dtype: str
insured_education_level --> <StringArray>
['College', 'Associate', 'Masters', 'MD', 'High School', 'JD', 'PhD']
Length: 7, dtype: str
insured_occupation --> <StringArray>
[     'armed-forces',      'adm-clerical', 'machine-op-inspct',
   'exec-managerial',     'other-service',   'farming-fishing',
    'prof-specialty', 'handlers-cleaners',   'priv-house-serv',
      'craft-repair',             'sales',  'transport-moving',
      'tech-support',   'protective-serv']
Length: 14, dtype: str
insured_hobbies --> <StringArray>
[      'exercise',        'camping',        'reading',     'basketball',
      'skydiving',           'golf',           'polo',          'chess',
       'yachting',   'base-jumping',    'board-games',      'paintball',
       'kayaking',       

C:\Users\asus\AppData\Local\Temp\ipykernel_17476\2391985237.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in X_train.select_dtypes(include= 'object').columns.to_list():


In [ ]:
for col in X_train.select_dtypes(exclude= 'object').columns.to_list():
    print (col,"-->" ,"min_value:",X_train[col].min(),", max_value:",X_train[col].max() )

months_as_customer --> min_value: 0 , max_value: 479
age --> min_value: 20 , max_value: 64
policy_deductable --> min_value: 500 , max_value: 2000
policy_annual_premium --> min_value: 433.33 , max_value: 2047.59
umbrella_limit --> min_value: -1000000 , max_value: 10000000
capital-gains --> min_value: 0 , max_value: 100500
capital-loss --> min_value: -111100 , max_value: 0
incident_hour_of_the_day --> min_value: 0 , max_value: 23
number_of_vehicles_involved --> min_value: 1 , max_value: 4
bodily_injuries --> min_value: 0 , max_value: 2
witnesses --> min_value: 0 , max_value: 3
injury_claim --> min_value: 0 , max_value: 21450
property_claim --> min_value: 0 , max_value: 23670
vehicle_claim --> min_value: 1440 , max_value: 79560
auto_year --> min_value: 1995 , max_value: 2015
policy_bind_year --> min_value: 1990 , max_value: 2015
policy_bind_month --> min_value: 1 , max_value: 12
policy_bind_day --> min_value: 1 , max_value: 31
incident_month --> min_value: 1 , max_value: 3
incident_day --

In [25]:
X_train.columns.to_list()

['months_as_customer',
 'age',
 'policy_state',
 'policy_csl',
 'policy_deductable',
 'policy_annual_premium',
 'umbrella_limit',
 'insured_sex',
 'insured_education_level',
 'insured_occupation',
 'insured_hobbies',
 'insured_relationship',
 'capital-gains',
 'capital-loss',
 'incident_type',
 'collision_type',
 'incident_severity',
 'authorities_contacted',
 'incident_state',
 'incident_city',
 'incident_hour_of_the_day',
 'number_of_vehicles_involved',
 'property_damage',
 'bodily_injuries',
 'witnesses',
 'police_report_available',
 'injury_claim',
 'property_claim',
 'vehicle_claim',
 'auto_make',
 'auto_model',
 'auto_year',
 'policy_bind_year',
 'policy_bind_month',
 'policy_bind_day',
 'incident_month',
 'incident_day']

In [ ]:
X_train['capital-gains'] = X_train['capital-gains'].replace('capital_gain')
X_train['capital-loss'] = X_train['capital-loss'].replace('capital_loss')

In [19]:
X_train_preprocessed = preprocessing.fit_transform(X_train)
X_test_preprocessed = preprocessing.transform(X_test)

In [20]:
df1.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 38 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   months_as_customer           1000 non-null   int64  
 1   age                          1000 non-null   int64  
 2   policy_state                 1000 non-null   str    
 3   policy_csl                   1000 non-null   str    
 4   policy_deductable            1000 non-null   int64  
 5   policy_annual_premium        1000 non-null   float64
 6   umbrella_limit               1000 non-null   int64  
 7   insured_sex                  1000 non-null   str    
 8   insured_education_level      1000 non-null   str    
 9   insured_occupation           1000 non-null   str    
 10  insured_hobbies              1000 non-null   str    
 11  insured_relationship         1000 non-null   str    
 12  capital-gains                1000 non-null   int64  
 13  capital-loss                 1

In [21]:
X_train.info()

<class 'pandas.DataFrame'>
Index: 800 entries, 887 to 594
Data columns (total 37 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   months_as_customer           800 non-null    int64  
 1   age                          800 non-null    int64  
 2   policy_state                 800 non-null    str    
 3   policy_csl                   800 non-null    str    
 4   policy_deductable            800 non-null    int64  
 5   policy_annual_premium        800 non-null    float64
 6   umbrella_limit               800 non-null    int64  
 7   insured_sex                  800 non-null    str    
 8   insured_education_level      800 non-null    str    
 9   insured_occupation           800 non-null    str    
 10  insured_hobbies              800 non-null    str    
 11  insured_relationship         800 non-null    str    
 12  capital-gains                800 non-null    int64  
 13  capital-loss                 800 n

In [23]:
X_train_preprocessed

array([[ 1.        ,  0.        ,  1.        , ...,  1.5309382 ,
         0.97922868,  0.4855868 ],
       [ 1.        ,  0.        ,  0.        , ..., -1.06196155,
        -0.94554514,  0.37014268],
       [ 1.        ,  0.        ,  0.        , ..., -0.0473486 ,
        -0.94554514, -0.32252199],
       ...,
       [ 0.        ,  0.        ,  1.        , ..., -1.40016586,
        -0.94554514,  1.29369558],
       [ 1.        ,  0.        ,  0.        , ..., -0.49828769,
        -0.94554514, -0.55341021],
       [ 0.        ,  1.        ,  1.        , ...,  0.40359048,
        -0.94554514, -0.66885432]], shape=(800, 141))